In [1]:
# Cell 2: Imports
import os
import re
import json
import base64
import uuid
from datetime import datetime, timezone
from pathlib import Path
from email import message_from_bytes
from bs4 import BeautifulSoup

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter,
    FieldCondition, MatchValue, MatchAny
)

from sentence_transformers import SentenceTransformer

print("✅ All imports successful")

c:\Users\RentoBees\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful


In [2]:
# Cell 3: Gmail Auth — upload credentials.json first via Colab file panel

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
CREDENTIALS_FILE = "credentials.json"
TOKEN_FILE = "token.json"

def get_gmail_service():
    """
    Authenticates with Gmail API using OAuth2.
    - First run: opens browser for login, saves token.json
    - Later runs: loads token.json silently, refreshes if expired
    Returns: authenticated Gmail API service object
    """
    creds = None

    # Load existing token if available
    if Path(TOKEN_FILE).exists():
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

    # If no valid credentials, run OAuth flow
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
            print("🔄 Token refreshed")
        else:
            if not Path(CREDENTIALS_FILE).exists():
                raise FileNotFoundError(
                    "credentials.json not found. "
                    "Upload it via the Colab file panel (left sidebar → Files)"
                )
            flow = InstalledAppFlow.from_client_secrets_file(
                CREDENTIALS_FILE, SCOPES
            )
            # In Colab, use run_console() — opens a URL you visit manually
            creds = flow.run_local_server(port=0)
            print("✅ Login successful")

        # Save token for future runs
        with open(TOKEN_FILE, "w") as f:
            f.write(creds.to_json())
        print(f"💾 Token saved to {TOKEN_FILE}")

    service = build("gmail", "v1", credentials=creds)
    print("✅ Gmail service ready")
    return service


# Test it
service = get_gmail_service()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=477657866532-mpobl3tafl0matv72mjo7u08t2l9pkbm.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A57891%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=vniq2O5L96bHh6ZPF0g9zHshbwPTe4&code_challenge=UHSSmneyC9dtDy7NIpZkgkZkMUS7RmYagaRiX3LL6ok&code_challenge_method=S256&access_type=offline
✅ Login successful
💾 Token saved to token.json
✅ Gmail service ready


In [3]:
# Test it
service = get_gmail_service()

✅ Gmail service ready


In [5]:
# Cell 4: Gmail Fetcher Functions

def fetch_email_list(service, max_results=10, label="INBOX"):
    """
    Fetches list of email IDs from Gmail inbox.
    Returns: list of { id, threadId }
    """
    try:
        result = service.users().messages().list(
            userId="me",
            maxResults=max_results,
            labelIds=[label]
        ).execute()

        messages = result.get("messages", [])
        print(f"📩 Found {len(messages)} emails")
        return messages

    except Exception as e:
        print(f"❌ Failed to fetch email list: {e}")
        return []


def fetch_email_by_id(service, email_id):
    """
    Fetches full email content by ID.
    Returns: raw Gmail API response dict
    """
    try:
        msg = service.users().messages().get(
            userId="me",
            id=email_id,
            format="full"
        ).execute()
        return msg

    except Exception as e:
        print(f"❌ Failed to fetch email {email_id}: {e}")
        return None


def fetch_thread_messages(service, thread_id):
    """
    Fetches all messages in a thread.
    Returns: list of raw Gmail API message dicts
    """
    try:
        thread = service.users().threads().get(
            userId="me",
            id=thread_id,
            format="full"
        ).execute()
        return thread.get("messages", [])

    except Exception as e:
        print(f"❌ Failed to fetch thread {thread_id}: {e}")
        return []


def fetch_attachment(service, email_id, attachment_id):
    """
    Fetches raw attachment bytes by attachment ID.
    Returns: bytes or None
    """
    try:
        att = service.users().messages().attachments().get(
            userId="me",
            messageId=email_id,
            id=attachment_id
        ).execute()

        data = att.get("data", "")
        return base64.urlsafe_b64decode(data)

    except Exception as e:
        print(f"❌ Failed to fetch attachment: {e}")
        return None




In [6]:
# Test
emails = fetch_email_list(service, max_results=5)
print(emails)

📩 Found 5 emails
[{'id': '1a18249019560fa3', 'threadId': '1a18249019560fa3'}, {'id': '1a181f9dcd495b22', 'threadId': '1a181f9dcd495b22'}, {'id': '1a18133a04aae63d', 'threadId': '1a18133a04aae63d'}, {'id': '1a18119813b1e4ff', 'threadId': '1a15d3e097b2126c'}, {'id': '1a180e8e94ba019a', 'threadId': '1a180e8e94ba019a'}]


In [7]:
# Cell 5: Gmail Parser — raw Gmail API response → clean structured dict

MIME_TO_FILETYPE = {
    "application/pdf": "pdf",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": "xlsx",
    "application/vnd.ms-excel": "xls",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": "docx",
    "application/msword": "doc",
    "text/csv": "csv",
    "image/png": "png",
    "image/jpeg": "jpg",
    "application/zip": "zip",
}

def _decode_base64(data: str) -> str:
    """Decode base64url encoded string to plain text."""
    try:
        return base64.urlsafe_b64decode(data + "==").decode("utf-8", errors="replace")
    except Exception:
        return ""

def _strip_html(html: str) -> str:
    """Strip HTML tags and return plain text."""
    try:
        return BeautifulSoup(html, "html.parser").get_text(separator=" ").strip()
    except Exception:
        return html

def _extract_parts(payload: dict, body_texts: list, attachments: list):
    """
    Recursively walks Gmail payload parts tree.
    Fills body_texts with plain text, attachments with metadata.
    """
    mime = payload.get("mimeType", "")
    body = payload.get("body", {})
    parts = payload.get("parts", [])

    if mime == "text/plain" and body.get("data"):
        body_texts.append(_decode_base64(body["data"]))

    elif mime == "text/html" and body.get("data") and not body_texts:
        # fallback: use HTML only if no plain text found yet
        body_texts.append(_strip_html(_decode_base64(body["data"])))

    elif mime not in ("text/plain", "text/html", "multipart/mixed",
                      "multipart/alternative", "multipart/related"):
        # it's an attachment
        filename = payload.get("filename", "")
        att_id = body.get("attachmentId")
        if att_id and filename:
            file_type = MIME_TO_FILETYPE.get(mime, mime.split("/")[-1])
            attachments.append({
                "filename": filename,
                "file_type": file_type,
                "attachment_id": att_id,
            })

    # recurse into child parts
    for part in parts:
        _extract_parts(part, body_texts, attachments)


def _extract_urls(text: str) -> list:
    """Extract URLs from text and infer their purpose."""
    url_pattern = re.compile(r'https?://[^\s<>"\']+')
    urls = url_pattern.findall(text)

    def infer_purpose(url: str) -> str:
        url_lower = url.lower()
        if any(k in url_lower for k in ["dashboard", "report", "analytics"]):
            return "dashboard or report link"
        elif any(k in url_lower for k in ["form", "survey", "response"]):
            return "form or survey"
        elif any(k in url_lower for k in ["drive", "docs", "sheet", "document"]):
            return "google drive document"
        elif any(k in url_lower for k in ["meet", "zoom", "teams", "calendar"]):
            return "meeting link"
        elif any(k in url_lower for k in ["unsubscribe", "optout"]):
            return "unsubscribe link"
        else:
            return "unknown"

    return [{"url": u, "purpose": infer_purpose(u)} for u in urls]


def _get_header(headers: list, name: str) -> str:
    """Get header value by name (case-insensitive)."""
    for h in headers:
        if h["name"].lower() == name.lower():
            return h["value"]
    return ""


def _parse_email_list(raw: str) -> list:
    """Parse comma-separated email addresses into a list."""
    if not raw:
        return []
    return [e.strip() for e in raw.split(",") if e.strip()]


def parse_email(raw_gmail_response: dict, thread_position: int = 1,
                thread_total: int = 1) -> dict | None:
    """
    Parses raw Gmail API response into clean structured dict
    matching the working_memory schema.

    Args:
        raw_gmail_response: full Gmail API message object
        thread_position: position of this message in its thread (1-indexed)
        thread_total: total messages in the thread

    Returns:
        Structured dict ready for working memory ingestion
    """
    if not raw_gmail_response:
        return None

    try:
        email_id  = raw_gmail_response.get("id", "")
        thread_id = raw_gmail_response.get("threadId", "")
        labels    = raw_gmail_response.get("labelIds", [])
        snippet   = raw_gmail_response.get("snippet", "")
        payload   = raw_gmail_response.get("payload", {})
        headers   = payload.get("headers", [])

        # ── People ────────────────────────────────────────────────────
        from_raw = _get_header(headers, "From")
        to_raw   = _get_header(headers, "To")
        cc_raw   = _get_header(headers, "Cc")

        people = {
            "from": from_raw,
            "to":   _parse_email_list(to_raw),
            "cc":   _parse_email_list(cc_raw),
        }

        # ── Email meta ────────────────────────────────────────────────
        subject = _get_header(headers, "Subject")
        date    = _get_header(headers, "Date")

        email_meta = {
            "subject": subject,
            "date":    date,
            "labels":  labels,
            "snippet": snippet,
        }

        # ── Body + Attachments ────────────────────────────────────────
        body_texts  = []
        attachments = []
        _extract_parts(payload, body_texts, attachments)
        body_text = "\n".join(body_texts).strip()

        # ── URLs ──────────────────────────────────────────────────────
        urls = _extract_urls(body_text)

        # ── Final record ──────────────────────────────────────────────
        return {
            "email_id":        email_id,
            "thread_id":       thread_id,
            "thread_position": thread_position,
            "thread_total":    thread_total,
            "people":          people,
            "email_meta":      email_meta,
            "body_text":       body_text,
            "attachments":     attachments,
            "urls":            urls,
            "memory_meta": {
                "ingested_at":   datetime.now(timezone.utc).isoformat(),
                "consolidated":  False,
                "ttl_hours":     48,
                "last_accessed": None,
            },
        }

    except Exception as e:
        print(f"❌ Failed to parse email {raw_gmail_response.get('id')}: {e}")
        return None



In [10]:

# ── Test ──────────────────────────────────────────────────────────────
emails = fetch_email_list(service, max_results=1)
if emails:
    raw = fetch_email_by_id(service, emails[0]["id"])
    parsed = parse_email(raw)
    print(json.dumps({
        k: parsed[k] for k in
        ["email_id", "thread_id", "people", "email_meta",
         "attachments", "urls"]
    }, indent=2))
    print(f"\n📝 Body preview: {parsed['body_text'][:300]}")

📩 Found 1 emails
{
  "email_id": "1a18249019560fa3",
  "thread_id": "1a18249019560fa3",
  "people": {
    "from": "Mayank Goyal <mayank.goyal@credresolve.com>",
    "to": [
      "prashanth.kumar@credresolve.com",
      "Manoj Kumar <manoj.kumar@credresolve.com>",
      "abhijit.das@credresolve.com"
    ],
    "cc": [
      "Aniket Saini <aniket.saini@credresolve.com>",
      "Manas Patra <manas.patra@credresolve.com>",
      "Somyashree Jena <somyashree.jena@credresolve.com>",
      "Praneeth Reddy <praneeth.reddy@credresolve.com>",
      "Sarthak Chandra <sarthak.chandra@credresolve.com>",
      "Abhishek Jha <Abhishek.jha@credresolve.com>",
      "Abhishek Sharma <abhishek.sharma@credresolve.com>",
      "Sameer.khan@credresolve.com",
      "Narva Siddhartha <siddhartha.narva@credresolve.com>",
      "Saujith Sathishkumar <saujith.sathishkumar@credresolve.com>"
    ]
  },
  "email_meta": {
    "subject": "Subject: HDB Financial Services \u2013 Vendor-wise QC Summary (Preliminary), N